<!-- Logos Banner -->
<div style="display: flex; align-items: center; justify-content: space-between; padding: 50px 0; border-bottom: 2px solid #003399;">
    <div style="display: flex; align-items: center;">
        <img src="https://raw.githubusercontent.com/geojupyter/jupytergis/main/packages/base/style/icons/logo.svg" alt="JupyterGIS" width="40" style="margin-right: 80px;">
        <img src="https://brand.esa.int/files/2020/05/ESA_logo_2020_Deep-scaled.jpg" alt="ESA" width="150">
    </div>
    <div style="display: flex; align-items: center;">
        <img src="https://quantstack.net/img/quantstack/logo-website.svg" alt="QuantStack" width="150" style="margin-right: 80px;">
        <img src="https://raw.githubusercontent.com/annefou/jupytergis-showcases/refs/heads/main/content/simula-logo.svg" alt="Simula" width="150" style="margin-right: 80px;">
        <img src="https://www.lifewatch.eu/wp-content/uploads/2021/07/logoLW_eric_outline2-01.svg" alt="Lifewatch ERIC" width="150">
    </div>
</div>

<!-- Badges -->
<p align="center">
    <a href="https://usegalaxy.eu/?tool_id=interactive_tool_jupytergis_notebook&version=latest"><img src="https://img.shields.io/badge/launch-Galaxy%20Europe-green?logo=jupyter" alt="Galaxy Europe"></a>
    <a href="https://github.com/geojupyter/jupytergis"><img src="https://img.shields.io/github/stars/geojupyter/jupytergis?style=social" alt="GitHub Stars"></a>
    <a href="https://jupytergis.readthedocs.io/"><img src="https://img.shields.io/badge/docs-readthedocs-blue" alt="Documentation"></a>
    <a href="https://opensource.org/licenses/BSD-3-Clause"><img src="https://img.shields.io/badge/License-BSD%203--Clause-blue.svg" alt="License"></a>
</p>

# Collaborative Annotation for ML Training Data

**Authors:** JupyterGIS Team  
**Affiliation:** QuantStack, Simula Research Laboratory, LifeWatch ERIC  
**Date:** December 2024  
**Contract:** ESA 4000144740/24/I-DT-bgh

---

## Purpose

This notebook demonstrates **real-time collaborative annotation** in JupyterGIS for creating machine learning training datasets:

- 👥 Multiple users annotating the same map simultaneously
- 📍 Geographically-referenced annotations with comments
- 💬 Threaded discussions on specific locations
- 🔄 Real-time synchronization across all collaborators

:::{important}
JupyterGIS is the **first GIS software** providing real-time collaborative editing features, similar to Google Docs but for geospatial data.
:::

## Use Case: Land Cover Classification

A common Earth Observation workflow requires **labeled training data** for machine learning:
```{mermaid}
flowchart LR
    A[Satellite Imagery] --> B[Human Labeling]
    B --> C[Training Dataset]
    C --> D[ML Model]
    D --> E[Land Cover Map]
```

:::{admonition} The Challenge
:class: warning

Traditional labeling workflows are:
- **Sequential** - One person labels, then passes to the next
- **Error-prone** - No real-time review or discussion
- **Slow** - Email/file sharing creates bottlenecks
:::

:::{admonition} The JupyterGIS Solution
:class: tip

With collaborative annotation:
- **Parallel** - Multiple experts label simultaneously
- **Reviewed** - Instant feedback and discussion
- **Fast** - All changes sync in real-time
:::

## Setup

First, we install and import the required libraries.

In [1]:
# Install pystac-client and myst and refresh your browser before running the next cells
!pip install pystac-client jupyterlab-myst -q

In [2]:
from jupytergis import GISDocument
from pystac_client import Client
import os
import json
from pyproj import Transformer
from collections import Counter

In [3]:
config_dir = "/home/jovyan/.jupyter"
os.makedirs(config_dir, exist_ok=True)

config_content = """{
  "ServerApp": {
    "collaborative": true
  }
}
"""

config_path = f"{config_dir}/jupyter_server_config.json"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"✅ Config created at: {config_path}")
print("⚠️  You must restart the Galaxy session for this to take effect!")
print("   (Not just kernel restart - full session restart)")

✅ Config created at: /home/jovyan/.jupyter/jupyter_server_config.json
⚠️  You must restart the Galaxy session for this to take effect!
   (Not just kernel restart - full session restart)


We'll create a map with satellite imagery as the base layer for annotation.

:::{note}
For this demo, we use **EOxCloudless** - a cloudless Sentinel-2 RGB mosaic from the Austrian company EOX. This high-resolution optical imagery is ideal for visual land cover annotation tasks.
:::

In [4]:
# Add ESA domains to the exempt list
current = os.environ.get("JGIS_EXEMPT_DOMAINS", "")
esa_domains = "https://eoresults.esa.int,https://browser.apex.esa.int"

if current:
    os.environ["JGIS_EXEMPT_DOMAINS"] = f"{current},{esa_domains}"
else:
    # Set with defaults + ESA
    os.environ["JGIS_EXEMPT_DOMAINS"] = f"https://geodes.cnes.fr,https://gdh-portal-prod.cnes.fr,https://geodes-portal.cnes.fr/api/stac/,{esa_domains}"

print("Updated JGIS_EXEMPT_DOMAINS:", os.environ.get("JGIS_EXEMPT_DOMAINS"))

Updated JGIS_EXEMPT_DOMAINS: https://geodes.cnes.fr,https://gdh-portal-prod.cnes.fr,https://geodes-portal.cnes.fr/api/stac/,https://eoresults.esa.int,https://browser.apex.esa.int


In [5]:
JGIS_FILE = "ML_Annotation_Demo.jGIS"
GEOJSON_FILE = "training_labels.geojson"

# Clean start - remove existing files
for f in [JGIS_FILE, GEOJSON_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️  Removed {f}")

print("✅ Ready for fresh demo")

🗑️  Removed ML_Annotation_Demo.jGIS
✅ Ready for fresh demo


## Load Satellite Imagery

We'll use the EOxCloudless Sentinel-2 mosaic, which provides:
- 🌍 Global cloudless coverage
- 📷 10m resolution RGB imagery  
- 🎨 Natural color visualization

This is representative of typical ML training workflows where annotators label visible features like forests, water bodies, settlements, and agricultural land.

## Create a map to collaborate

In [6]:
# Create the annotation workspace
doc = GISDocument("ML_Annotation_Demo.jGIS", latitude=47.0, longitude=11.0, zoom=10)

doc.add_raster_layer(
    url="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    name="OpenStreetMap"
)

doc.add_tiff_layer(
    url="https://s2downloads.eox.at/demo/EOxCloudless/2020/rgbnir/s2cloudless2020-16bits_sinlge-file_z0-4.tif",
    name="Sentinel-2 Cloudless (2020)",
    min=0,
    max=1000,
    opacity=0.9
)

'14668cd4-3a30-40a8-84b6-e106e0ff87f3'

In [7]:
doc

## How to Add Annotations

:::{admonition} Step-by-Step Guide
:class: tip

1. **Right-click** anywhere on the map
2. Select **"Add Annotation"** from the context menu
3. Enter your annotation text (e.g., "Class X")
4. Click **OK** to save

The annotation appears as a marker with your comment attached.
:::

### Annotation Panel

The **Annotations Panel** (right sidebar) shows all annotations:

| Icon | Action |
|------|--------|
| 📍 | Center map on annotation |
| 💬 | View/add comments |
| 🗑️ | Delete annotation |

:::{note}
Each annotation stores:
- **Geographic coordinates** (latitude, longitude)
- **Author** information
- **Timestamp**
- **Threaded comments**
:::

## Real-Time Collaboration

:::{important}
To collaborate in real-time, multiple users must open the **same `.jGIS` file** on a platform with Real-Time Collaboration (RTC) enabled.
:::

### Supported Platforms

| Platform | RTC Enabled | How to Share |
|----------|-------------|--------------|
| **Galaxy Europe** | ✅ Yes | Share session URL |
| **JupyterHub** | ✅ Yes | Share file path |
| **Local JupyterLab** | ⚠️ Requires setup | Use jupyter-collaboration |
| **CDSE** | ❌ Disabled | Not available |

### What Syncs in Real-Time

- ✅ Annotations added/edited/deleted
- ✅ Comments and replies
- ✅ Layer visibility and opacity
- ✅ Map view (pan/zoom)
- ✅ Cursor positions of collaborators

## ML Training Data Workflow

:::{admonition} Example: Alpine Land Cover Classification
:class: example

**Team Setup:**
- 👩‍🔬 **Expert A:** Labels forests and vegetation
- 👨‍🔬 **Expert B:** Labels water bodies and glaciers
- 👩‍💻 **Expert C:** Labels settlements and infrastructure
- 👨‍💼 **Reviewer:** Validates annotations

**Annotation Labels for Alpine Regions:**
```
🌲 Forest          → "forest"
🌿 Alpine meadow   → "meadow"
💧 Lake/River      → "water"
❄️ Glacier/Snow    → "glacier"
🏘️ Settlement      → "urban"
🪨 Bare rock       → "rock"
🌾 Agriculture     → "agriculture"
```
:::

:::{tip}
Use consistent labeling conventions across your team. The RGB imagery makes it easy to visually distinguish different land cover types.
:::

## Exercise: Create Sample Annotations

:::{exercise}
:label: annotation-exercise

Open the `Annotation_Demo.jGIS` file from the file browser and add at least 3 annotations:

1. Find a **forested area** (dark green) and annotate it
2. Find a **lake or river** (dark blue/black) and annotate it
3. Find a **settlement or village** (gray/brown clusters) and annotate it

For each annotation, include:
- A descriptive label (e.g., "Coniferous forest")
- Brief justification for the classification
:::

:::{solution} annotation-exercise
:class: dropdown

Example annotations in the Austrian Alps:

**Annotation 1:** 
- Location: Slopes near Innsbruck
- Text: "Dense coniferous forest - dark green, uniform texture"

**Annotation 2:**
- Location: Achensee
- Text: "Alpine lake - dark blue, clear boundaries"

**Annotation 3:**
- Location: Small village in valley
- Text: "Rural settlement - clustered buildings, road network visible"
:::

## Extract Annotations

After adding annotations in the editor, we can extract them programmatically for ML training.

In [8]:
def extract_annotations(jgis_file):
    """
    Extract annotations from a JupyterGIS file.
    
    Parameters
    ----------
    jgis_file : str
        Path to the .jGIS file
        
    Returns
    -------
    list
        List of annotation dictionaries
    """
    with open(jgis_file, "r") as f:
        data = json.load(f)
    
    annotations = []
    for key, value in data.get("metadata", {}).items():
        if key.startswith("annotation_"):
            annotations.append({"id": key, **value})
    
    return annotations


def annotations_to_geojson(annotations, source_crs="EPSG:3857", target_crs="EPSG:4326"):
    """
    Convert JupyterGIS annotations to GeoJSON format.
    
    Parameters
    ----------
    annotations : list
        List of annotation dictionaries from extract_annotations()
    source_crs : str
        Source coordinate reference system (default: Web Mercator)
    target_crs : str
        Target coordinate reference system (default: WGS84)
        
    Returns
    -------
    dict
        GeoJSON FeatureCollection
    """
    transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)
    
    features = []
    for ann in annotations:
        # Transform coordinates
        lon, lat = transformer.transform(
            ann["position"]["x"], 
            ann["position"]["y"]
        )
        
        # Collect all labels from contributors
        labels = [c["value"] for c in ann.get("contents", [])]
        contributors = [c["user"]["display_name"] for c in ann.get("contents", [])]
        
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [lon, lat]
            },
            "properties": {
                "id": ann["id"],
                "labels": labels,
                "primary_label": labels[0] if labels else None,
                "contributors": contributors,
                "num_contributors": len(contributors),
                "zoom_level": ann.get("zoom")
            }
        }
        features.append(feature)
    
    return {
        "type": "FeatureCollection",
        "features": features
    }

## Load and Inspect Annotations

Now let's extract the annotations you created:

In [9]:
# Extract annotations from the saved file
annotations = extract_annotations("ML_Annotation_Demo.jGIS")

print(f"📍 Found {len(annotations)} annotations\n")

# Display each annotation
for ann in annotations:
    labels = [c["value"] for c in ann.get("contents", [])]
    contributors = [c["user"]["display_name"] for c in ann.get("contents", [])]
    
    print(f"Annotation: {ann['id'][-8:]}")
    print(f"  Position (EPSG:3857): x={ann['position']['x']:.2f}, y={ann['position']['y']:.2f}")
    print(f"  Labels: {labels}")
    print(f"  Contributors: {contributors}")
    print()

📍 Found 0 annotations



## Export to GeoJSON

Convert annotations to standard GeoJSON format for use in ML pipelines:

In [10]:
# Convert to GeoJSON
geojson = annotations_to_geojson(annotations)

# Display the GeoJSON
print("GeoJSON FeatureCollection:")
print(json.dumps(geojson, indent=2))

GeoJSON FeatureCollection:
{
  "type": "FeatureCollection",
  "features": []
}


In [11]:
# Save for ML pipeline
output_file = "training_labels.geojson"

with open(output_file, "w") as f:
    json.dump(geojson, f, indent=2)

print(f"✅ Exported {len(geojson['features'])} annotations to {output_file}")

# Show label distribution
print("\n📊 Label Distribution:")
all_labels = [f["properties"]["primary_label"] for f in geojson["features"]]
for label, count in Counter(all_labels).items():
    print(f"   {label}: {count}")

✅ Exported 0 annotations to training_labels.geojson

📊 Label Distribution:


## Visualize Training Labels

Add the exported labels back to the map as a vector layer to verify the annotations:

In [12]:
# Reload the document and add training labels as a layer
doc = GISDocument("ML_Annotation_Demo-annotated.jGIS")

# Add the GeoJSON as a vector layer
doc.add_geojson_layer(
    path="training_labels.geojson",
    name="Training Labels"
)

doc

## Next Steps: ML Pipeline Integration

The exported `training_labels.geojson` can be used in standard ML workflows:

:::{admonition} Example: Extracting Training Samples
:class: example
```python
import rasterio
import geopandas as gpd

# Load labels
labels = gpd.read_file("training_labels.geojson")

# Load satellite imagery
with rasterio.open("satellite_image.tif") as src:
    # Extract pixel values at label locations
    coords = [(p.x, p.y) for p in labels.geometry]
    samples = [x[0] for x in src.sample(coords)]

# Create training dataset
X = np.array(samples)  # Features (pixel values)
y = labels["primary_label"].values  # Labels
```
:::

:::{seealso}
Common ML frameworks for land cover classification:
- **scikit-learn**: Random Forest, SVM
- **TensorFlow/Keras**: CNN, U-Net
- **PyTorch**: ResNet, DeepLab
:::

## Collaboration Features

JupyterGIS annotations support real-time collaboration:

| Feature | Description |
|---------|-------------|
| **Multi-user** | Multiple experts annotate simultaneously |
| **User tracking** | Each annotation records contributor info |
| **Comments** | Threaded discussions on each point |
| **Consensus** | Multiple labels per point for agreement analysis |

:::{tip}
For quality control, you can analyze inter-annotator agreement:
```python
# Check annotations with multiple contributors
multi_label = [f for f in geojson["features"] if f["properties"]["num_contributors"] > 1]
print(f"Annotations with multiple reviewers: {len(multi_label)}")
```
:::

## Summary

:::{list-table} Complete Workflow
:header-rows: 1

* - Step
  - Tool
  - Output
* - 1. Create workspace
  - Notebook + Python API
  - `.jGIS` file with imagery
* - 2. Add annotations
  - JupyterGIS Editor
  - Annotations in metadata
* - 3. Extract annotations
  - Notebook + Python
  - Annotation list
* - 4. Export to GeoJSON
  - Python
  - `training_labels.geojson`
* - 5. ML training
  - scikit-learn / TensorFlow
  - Trained model
:::

:::{attention}
**Key Advantages of JupyterGIS for ML Workflows:**
- 🔄 Seamless notebook ↔ editor workflow
- 👥 Real-time collaboration for team labeling
- 📍 Georeferenced annotations with CRS handling
- 📤 Standard GeoJSON export for any ML framework
:::

## References

:::{seealso}
**Data Sources:**
- [EOxCloudless](https://s2maps.eu/) - Sentinel-2 cloudless mosaic
- [OpenStreetMap](https://www.openstreetmap.org/)

**Documentation:**
- [JupyterGIS Documentation](https://jupytergis.org)
- [GeoJSON Specification](https://geojson.org/)

**Related Tools:**
- [Label Studio](https://labelstud.io/) - General purpose labeling
- [CVAT](https://cvat.ai/) - Computer vision annotation
:::

---

*This demo was developed as part of ESA Contract 4000144740/24/I-DT-bgh*